In [0]:
from pyspark.sql.functions import col, to_date, date_format, when, trim, regexp_replace
from pyspark.sql.types import DateType
from datetime import datetime
from pyspark.sql.functions import udf

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_t1 = spark.table("bronze.lbtt_table_1")

def parse_lbtt_date(s):
    if s is None:
        return None
    try:
        clean = s.strip().split(" ")[0]  # "Apr-15 F" -> "Apr-15"
        return datetime.strptime(clean, "%b-%y").date()
    except:
        return None

parse_date_udf = udf(parse_lbtt_date, DateType())

df_t1_silver = (
    df_t1
    .withColumn("is_provisional", when(col("month").contains(" P"), True).otherwise(False))
    .withColumn("report_date", parse_date_udf(col("month")))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .drop("month")
    .select(
        "report_date", "year_month", "is_provisional",
        "residential_returns", "residential_tax_ex_ads",
        "residential_ads", "residential_tax_total",
        "non_residential_returns", "non_residential_tax_ex_ads",
        "non_residential_ads", "non_residential_tax_total",
        "total_returns", "total_tax_ex_ads", "total_ads", "total_tax"
    )
)

df_t1_silver.show(5)
df_t1_silver.printSchema()

In [0]:
(
    df_t1_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.lbtt_table_1")
)

In [0]:
df_t2 = spark.table("bronze.lbtt_table_2")

df_t2_silver = (
    df_t2
    .withColumn("is_provisional", when(col("month").contains(" P"), True).otherwise(False))
    .withColumn("report_date", parse_date_udf(col("month")))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .drop("month")
    .select(
        "report_date", "year_month", "is_provisional",
        "band_up_to_145k_returns", "band_up_to_145k_tax",
        "band_145k_to_250k_returns", "band_145k_to_250k_tax",
        "band_250k_to_325k_returns", "band_250k_to_325k_tax",
        "band_325k_to_750k_returns", "band_325k_to_750k_tax",
        "band_over_750k_returns", "band_over_750k_tax",
        "total_returns", "total_tax_ex_ads"
    )
)

df_t2_silver.show(5)
df_t2_silver.printSchema()

In [0]:
(
    df_t2_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.lbtt_table_2")
)